# Earlier findings of pasquereau-2013-02, checked on DANDI:001636 version 0.261124.1906

Paper: Pasquereau B, Turner RS (2013). Primary motor cortex of the parkinsonian monkey: altered
neuronal responses to muscle stretch. *Front. Syst. Neurosci.* 7:98,
[doi:10.3389/fnsys.2013.00098](https://doi.org/10.3389/fnsys.2013.00098).

Dataset: [DANDI:001636](https://dandiarchive.org/dandiset/001636/0.261124.1906), version
`0.261124.1906`.

Attempt pasquereau-2013-02 read the draft of the dandiset (modified 2026-10-21) and recorded eight
findings. Findings 1, 2, 4 and 5 are reported again by pasquereau-2013-03 (its findings 1, 3, 7 and
5), and its notebook `reproduce_pasquereau_2013.ipynb` shows them. This notebook checks the other
four, 3, 6, 7 and 8, on this version. Each section opens with the earlier statement.

In [ ]:
import json
import urllib.request

import h5py
import numpy as np
import pandas as pd
import remfile
from dandi.dandiapi import DandiAPIClient
from scipy import stats

import pasq

pd.set_option("display.width", 160)
venus_urls = pasq.asset_urls("sub-Venus")
leu_urls = pasq.asset_urls("sub-Leu")
with DandiAPIClient() as client:
    meta = client.get_dandiset(pasq.DANDISET_ID, pasq.DANDISET_VERSION).get_raw_metadata()
print(f"{len(venus_urls)} Venus files, {len(leu_urls)} Leu files at version {pasq.DANDISET_VERSION}, "
      f"published {meta['datePublished'][:10]}")

## Finding 3

Earlier statement: "Partly fixed since pasquereau-2013-01. The EMG description now gives the paper's
processing (rectified, low-pass filtered at 100 Hz) where it said sample-hold integrated; the values
are unchanged. The monkey L analog streams are still sampled at 100 Hz (the paper says 200 Hz) and
the EMG is in uncalibrated A/D units, both now stated in the dandiset's description."

The paper's Methods: analog data "were digitized at either 200-Hz (monkey L) or 500-Hz (monkey V)".
Below, the sampling interval of every analog series in every Leu file, the unit and description of
the EMG series, and the dandiset description's sentences on both.

In [ ]:
def analog_summary(path, url):
    with h5py.File(remfile.File(url), "r") as h:
        acq = h["acquisition"]
        rows = []
        for k in acq:
            g = acq[k]
            if "data" not in g or "timestamps" not in g:
                continue
            ts = g["timestamps"][:1000]
            rows.append({"file": path.split("/")[-1], "series": k,
                         "interval_ms": round(float(np.median(np.diff(ts))) * 1000, 3),
                         "unit": g["data"].attrs.get("unit"),
                         "description": g.attrs.get("description", "")})
        return rows


leu_analog = pd.DataFrame([r for pu in leu_urls for r in analog_summary(*pu)])
print("Leu files with an analog series:", leu_analog.file.nunique(), "of", len(leu_urls))
print("Leu analog series by name and sampling interval (ms):")
print(leu_analog.groupby(["series", "interval_ms"]).size().to_string())
emg = leu_analog[leu_analog.series.str.startswith("EMG")]
print("Leu EMG series:", len(emg), "| units:", emg.unit.value_counts().to_dict())
venus_one = pd.DataFrame(analog_summary(*venus_urls[0]))
print("\nVenus file", venus_urls[0][0].split("/")[-1])
print(venus_one[["series", "interval_ms", "unit"]].to_string(index=False))
print("\nDescription of", venus_one.series.iloc[0], ":", venus_one.description.iloc[0])
print("\nDandiset description, sentences on the sampling rate and units:")
for sentence in meta["description"].split(". "):
    if "100 Hz" in sentence or "arbitrary units" in sentence:
        print(" ", sentence.strip())

## Finding 6

Earlier statement: "Still present. The pre-MPTP PTN and CSN resting rates are 16.3 and 3.6 spikes/s
in the text and 18.3 and 3.2 in Table 1. Table 1 marks the CSN change in the fraction of excitatory
responses p < 0.01, but its own counts (23 of 24 against 19 of 22) give p = 0.54 with the
Yates-corrected chi-square test that reproduces its other chi-square values."

This is about the paper, which has not changed. The text (Results) gives "mean rate: 16.3-spikes/s"
for PTNs and "mean rate: 3.6-spikes/s" for CSNs, citing Table 1; Table 1 gives 18.3 ± 8.6 and
3.2 ± 4.0. Table 1's CSN ratio of excitatory responses is 13/13 and 10/11 before MPTP and 11/13 and
8/9 after, marked "## p < 0.01 (χ2 test)". The test on those counts:

In [ ]:
text_rates = {"PTN": 16.3, "CSN": 3.6}
table_rates = {"PTN": 18.3, "CSN": 3.2}
print("pre-MPTP resting rates, text against Table 1:", {k: (text_rates[k], table_rates[k]) for k in text_rates})
csn = np.array([[13 + 10, 13 + 11 - (13 + 10)],    # pre-MPTP: excitatory, inhibitory
                [11 + 8, 13 + 9 - (11 + 8)]])      # post-MPTP
for correction in (True, False):
    chi2, p, dof, _ = stats.chi2_contingency(csn, correction=correction)
    print(f"CSN excitatory {csn[0, 0]}/{csn[0].sum()} against {csn[1, 0]}/{csn[1].sum()}, "
          f"Yates correction {correction}: chi2 = {chi2:.3f}, p = {p:.2f}")
print(f"Fisher exact p = {stats.fisher_exact(csn)[1]:.2f}")

## Finding 7

Earlier statement: "New. Three monkey V sessions (v1401, v1402, v1501) have 51 labelled perturbations
but no pulse in the torque command, as the draft's torque description says. Their perturbations are
kept here, as the command is not used to select trials."

The torque command's pulse size of each labelled perturbation in these three files (`pasq.command_size`:
mean over 10 to 50 ms after onset less the mean over the 50 ms before, signed by direction), against
the same measure in the Venus files just before and after each of them in session order, and the files'
torque description.

In [ ]:
codes = ["v1401", "v1402", "v1501"]
named = [pu for pu in venus_urls if any(f"++{c}++" in pu[0] for c in codes)]
index = sorted({j for i, pu in enumerate(venus_urls) if pu in named for j in (i - 1, i + 1)})
neighbours = [venus_urls[j] for j in index if venus_urls[j] not in named]
rows = []
for group, pus in (("named", named), ("neighbour", neighbours)):
    for x in pasq.load_all(pus):
        c = pasq.command_size(x)
        span = np.nanmax(x["torque_cmd"], axis=1) - np.nanmin(x["torque_cmd"], axis=1)
        rows.append({"group": group, "session": x["session"].split("++")[1], "perturbations": len(x["pert"]),
                     "median_pulse": round(float(np.nanmedian(c)), 1),
                     "max_abs_pulse": round(float(np.nanmax(np.abs(c))), 1),
                     "max_span_-50_to_100_ms": round(float(np.nanmax(span)), 1)})
pulses = pd.DataFrame(rows)
print(pulses.to_string(index=False))
print("labelled perturbations in the three named sessions:",
      int(pulses[pulses.group == "named"].perturbations.sum()))
with h5py.File(remfile.File(named[0][1]), "r") as h:
    desc = h["acquisition/ElbowTorque"].attrs["description"]
print("\nElbowTorque description, the sentence that names the sessions:")
print(" ", [s for s in desc.split(". ") if "v1401" in s][0])

## Finding 8

Earlier statement: "New. The dandiset names catalystneuro/turner-lab-to-nwb as the code that compiled
it, but the repository's main branch is still at e80a192917490b50db6871180b6d1d9a24772e35, read for
pasquereau-2013-01, and does not hold the code that built the draft (merged sessions, obs_intervals,
the new unit names and descriptions). That code is in open pull request 33, branch fix_errors at
5f9bf6320a0160e4e56301d5409fb9242c1d8cf4."

The dandiset's `IsCompiledBy` resource at this version, and the state of pull request 33 and of the
repository's main branch from the GitHub API, read when this notebook runs.

In [ ]:
def github(path):
    req = urllib.request.Request("https://api.github.com/repos/catalystneuro/turner-lab-to-nwb/" + path,
                                 headers={"Accept": "application/vnd.github+json"})
    with urllib.request.urlopen(req, timeout=60) as r:
        return json.load(r)


compiled = [r for r in meta["relatedResource"] if r["relation"] == "dcite:IsCompiledBy"]
print("IsCompiledBy:", [(r["name"], r["url"]) for r in compiled])
pr = github("pulls/33")
print(f"pull request 33 ({pr['head']['ref']}): state {pr['state']}, merged at {pr['merged_at']}, "
      f"merge commit {pr['merge_commit_sha']}")
cmp_ = github(f"compare/{pr['merge_commit_sha']}...main")
print(f"main against the merge commit: {cmp_['status']} (ahead by {cmp_['ahead_by']}, behind by {cmp_['behind_by']})")
print("version published", meta["datePublished"])